# 1. Your first pipeline

Load clinical notes, de-identify them, split them into sections and chunk them,
then look at the **manifest** every run produces.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## Some notes

Two short synthetic notes in a temporary folder.

In [2]:
import os
import pathlib
import tempfile

# Work in a scratch folder so nothing is written next to this notebook.
workdir = pathlib.Path(tempfile.mkdtemp())
os.chdir(workdir)

notes = pathlib.Path("notes")
notes.mkdir()

(notes / "note1.txt").write_text(
    "Chief Complaint:\n"
    "Chest pain for two days.\n"
    "Assessment:\n"
    "Likely musculoskeletal. Patient SSN 123-45-6789 on file.\n"
    "Plan:\n"
    "Ibuprofen; follow up in one week.\n",
    encoding="utf-8",
)
(notes / "note2.txt").write_text(
    "Chief Complaint:\n"
    "Cough and fever.\n"
    "Plan:\n"
    "Chest X-ray; call (555) 010-2345 with results.\n",
    encoding="utf-8",
)
sorted(p.name for p in notes.iterdir())

['note1.txt', 'note2.txt']

## Build and run the pipeline

Components are addressed by a permanent string key such as
`loader.clinical_text.plain_text`. Importing `openbtk.data.clinical_text`
registers them.

In [3]:
from openbtk.data import clinical_text  # noqa: F401  (registers the components)
from openbtk.pipelines import Pipeline, Step

pipeline = (
    Pipeline("first-pipeline")
    .add(Step("load", "loader.clinical_text.plain_text", path=str(notes)))
    .add(Step("deid", "preprocessor.general.deidentify", mode="redact"))
    .add(Step("segment", "preprocessor.clinical_text.section_segment"))
    .add(Step("chunk", "chunker.clinical_text.section_aware", max_tokens=40))
)
manifest = pipeline.run()
manifest.status

'success'

## The manifest

`run()` returns a `RunManifest` and **never** the processed records: it is the
audit record. Per step it says what ran and how many records went in and out.

In [4]:
for step in manifest.steps:
    name = step.component.class_name
    print(f"{step.step_id:8} {name:26} {step.records_in} in -> {step.records_out} out")

load     PlainTextLoader            0 in -> 2 out
deid     DeidPreprocessor           2 in -> 2 out
segment  SectionSegmenter           2 in -> 2 out
chunk    SectionAwareChunker        2 in -> 5 out


In [5]:
dump = manifest.model_dump_json(indent=2)

# No note content, and none of the identifiers we planted, is in the manifest.
assert (
    "123-45-6789" not in dump
    and "010-2345" not in dump
    and "chest pain" not in dump.lower()
)
print(dump[:520], "...")

{
  "manifest_version": "1.0.0",
  "run_id": "86af26f42cf745cb9d713ec19a5a2a9c",
  "status": "success",
  "config": {
    "name": "first-pipeline",
    "version": 1,
    "description": null,
    "policy": {
      "allow_offsite_providers": false,
      "fail_on_guardrail_block": true
    },
    "provenance": {
      "manifest_dir": "./runs"
    },
    "steps": [
      {
        "id": "load",
        "type": "loader.clinical_text.plain_text",
        "params": {
          "path": "notes"
        },
        "after":  ...


## The same run from a config file

A pipeline is data, so it can be reviewed and diffed. The `openbtk` command
line validates and runs a YAML config; here we call the same code in-process.

In [6]:
from openbtk.cli import main

config = pathlib.Path("pipeline.yaml")
config.write_text(
    """
name: first-pipeline
provenance:
  manifest_dir: runs
steps:
  - id: load
    type: loader.clinical_text.plain_text
    params: {path: notes}
  - id: deid
    type: preprocessor.general.deidentify
    params: {mode: redact}
    after: [load]
  - id: chunk
    type: chunker.clinical_text.section_aware
    params: {max_tokens: 40}
    after: [deid]
""",
    encoding="utf-8",
)

print("validate ->", main(["validate", str(config)]))  # 0 means valid; nothing was run
print("run      ->", main(["run", str(config)]))

OK: pipeline.yaml (3 step(s)); nothing was run.
validate -> 0
run da20e750b1f64be78452efd872b4f04e: success
  load: 0 in -> 2 out
  deid: 2 in -> 2 out
  chunk: 2 in -> 2 out
manifest: runs\da20e750b1f64be78452efd872b4f04e.json
run      -> 0


## Replay it

`replay` rebuilds the pipeline from the manifest, runs it again, and compares
input content and per-step counts. Exit code `0` means it matched.

In [7]:
manifest_file = next(pathlib.Path("runs").glob("*.json"))
print("replay   ->", main(["replay", str(manifest_file)]))

run b6eaa0e22e4647f0bea2168829ecd7df: success
  load: 0 in -> 2 out
  deid: 2 in -> 2 out
  chunk: 2 in -> 2 out
manifest: runs\b6eaa0e22e4647f0bea2168829ecd7df.json
replay matches the recorded run.
note: input notes has no content hash (a directory or a very large file), so only its record count was compared.
replay   -> 0


## Where next

* [De-identification](02_deidentification.ipynb) - modes, the audit report, and
  measuring how well it works.
* The [command-line guide](https://openbtk.org/openbtk-core/dev/guides/cli/).